# 🍈 DurianVision — Training Classifier (Level 3: Stage 2)

Notebook ini melatih **EfficientNet-V2-S** sebagai classifier varietas durian.

### Arsitektur Two-Stage:
```
Frame → YOLO (detect durian) → Crop → EfficientNet (classify variety) → Result
```

### Keunggulan:
- YOLO hanya perlu mendeteksi "ini durian atau bukan" (mudah)
- EfficientNet spesialis membedakan varietas dari crop close-up (jauh lebih akurat)
- Bisa menambah varietas baru tanpa retrain YOLO

### Persiapan:
1. Runtime → **GPU (T4)**
2. Upload `dataset.zip` yang sama ke Google Drive (sudah ada jika sudah menjalankan notebook Level 2)

---

## Langkah 0: Setup Environment

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1024**3:.1f} GB)")
else:
    raise RuntimeError("GPU diperlukan! Runtime → Change runtime type → GPU")

!pip install -q albumentations timm
print("\n✅ Ready")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==========================================
# KONFIGURASI — UBAH SESUAI KEBUTUHAN
# ==========================================

DATASET_ZIP = "/content/drive/MyDrive/dataset.zip"  # Sama dengan notebook Level 2
OUTPUT_DRIVE = "/content/drive/MyDrive/DurianVision_Training"
WORK_DIR = "/content/classifier_training"

# Classifier config
MODEL_ARCH = "efficientnet_v2_s"  # Pilihan: efficientnet_v2_s, resnet50, mobilenet_v3_large
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
IMAGE_SIZE = 224
NUM_WORKERS = 2

print(f"Model: {MODEL_ARCH}, Epochs: {EPOCHS}, Batch: {BATCH_SIZE}, ImgSize: {IMAGE_SIZE}")

## Langkah 1: Ekstrak Crop dari Dataset YOLO

Mengambil setiap bounding box dari label YOLO, crop dari gambar asli,
lalu simpan ke folder per kelas (format ImageFolder untuk PyTorch).

In [ ]:
import os, zipfile, shutil, yaml, cv2
import collections
from pathlib import Path

# Setup dirs
DATASET_DIR = f"{WORK_DIR}/dataset"
CROPS_DIR = f"{WORK_DIR}/crops"

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)

# Extract dataset
print(f"📦 Extracting {DATASET_ZIP}...")
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall(DATASET_DIR)

# Find data.yaml
data_yaml = os.path.join(DATASET_DIR, 'data.yaml')
if not os.path.exists(data_yaml):
    for root, dirs, files in os.walk(DATASET_DIR):
        if 'data.yaml' in files:
            actual = root
            if actual != DATASET_DIR:
                for item in os.listdir(actual):
                    shutil.move(os.path.join(actual, item), os.path.join(DATASET_DIR, item))
            break

with open(os.path.join(DATASET_DIR, 'data.yaml')) as f:
    config = yaml.safe_load(f)

yolo_names = config['names']

# Map YOLO class names to UI class names (same as in the app)
CLASS_MAP = {
    'bawor': 'Bawor',
    'black thorn': 'Black Thorn',
    'kanyao': 'Kani',
    'monthong': 'Monthong',
    'musang king': 'Musang King',
    'not durian': 'Lainnya',
}

# Resolve class names
CLASS_NAMES = []
for name in yolo_names:
    mapped = CLASS_MAP.get(name.lower(), name)
    CLASS_NAMES.append(mapped)

print(f"\nKelas: {CLASS_NAMES}")
print(f"Jumlah kelas: {len(CLASS_NAMES)}")

In [ ]:
from tqdm import tqdm
import numpy as np

def extract_crops(dataset_dir, crops_dir, class_names, split='train', padding=0.1):
    """
    Extract crops from YOLO dataset.
    Reads each label file, crops the bbox region from the image,
    saves to crops_dir/split/class_name/
    """
    img_dir = os.path.join(dataset_dir, split, 'images')
    lbl_dir = os.path.join(dataset_dir, split, 'labels')

    if not os.path.exists(img_dir) or not os.path.exists(lbl_dir):
        print(f"  ⚠️ {split} tidak ditemukan, skip")
        return {}

    # Create output dirs
    out_split = 'val' if split == 'valid' else split
    for name in class_names:
        os.makedirs(os.path.join(crops_dir, out_split, name), exist_ok=True)

    counts = collections.Counter()
    label_files = [f for f in os.listdir(lbl_dir) if f.endswith('.txt')]

    for lbl_file in tqdm(label_files, desc=f"  Extracting {split}"):
        base = os.path.splitext(lbl_file)[0]

        # Find image
        img_path = None
        for ext in ['.jpg', '.jpeg', '.png', '.JPG', '.PNG']:
            candidate = os.path.join(img_dir, base + ext)
            if os.path.exists(candidate):
                img_path = candidate
                break
        if img_path is None:
            continue

        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        with open(os.path.join(lbl_dir, lbl_file)) as f:
            lines = f.readlines()

        for li, line in enumerate(lines):
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls_id = int(parts[0])
            if cls_id >= len(class_names):
                continue

            cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

            # Convert YOLO format to pixel coords
            x1 = int((cx - bw/2) * w)
            y1 = int((cy - bh/2) * h)
            x2 = int((cx + bw/2) * w)
            y2 = int((cy + bh/2) * h)

            # Add padding
            pw = int((x2 - x1) * padding)
            ph = int((y2 - y1) * padding)
            x1 = max(0, x1 - pw)
            y1 = max(0, y1 - ph)
            x2 = min(w, x2 + pw)
            y2 = min(h, y2 + ph)

            crop = img[y1:y2, x1:x2]
            if crop.size == 0 or crop.shape[0] < 10 or crop.shape[1] < 10:
                continue

            class_name = class_names[cls_id]
            counts[class_name] += 1

            out_split = 'val' if split == 'valid' else split
            out_path = os.path.join(crops_dir, out_split, class_name,
                                   f"{base}_crop{li}.jpg")
            cv2.imwrite(out_path, crop)

    return dict(counts)

print("🔪 Extracting crops dari YOLO dataset...\n")

train_counts = extract_crops(DATASET_DIR, CROPS_DIR, CLASS_NAMES, 'train')
val_counts = extract_crops(DATASET_DIR, CROPS_DIR, CLASS_NAMES, 'valid')
test_counts = extract_crops(DATASET_DIR, CROPS_DIR, CLASS_NAMES, 'test')

print(f"\n📊 Crop distribution:")
print(f"\n  TRAIN:")
for name, n in sorted(train_counts.items(), key=lambda x: -x[1]):
    print(f"    {name:15s}: {n}")
print(f"    Total: {sum(train_counts.values())}")

if val_counts:
    print(f"\n  VAL:")
    for name, n in sorted(val_counts.items(), key=lambda x: -x[1]):
        print(f"    {name:15s}: {n}")
    print(f"    Total: {sum(val_counts.values())}")

## Langkah 2: Augmentasi & Balance Crops

Menyeimbangkan jumlah crop per kelas via augmentasi offline, lalu visualisasi sampel.

In [ ]:
import albumentations as A
import random
import glob

def balance_crops(crops_dir, split='train'):
    """Balance crop counts per class via augmentation."""
    split_dir = os.path.join(crops_dir, split)
    if not os.path.exists(split_dir):
        return

    # Count per class
    class_counts = {}
    for cls_dir in sorted(os.listdir(split_dir)):
        cls_path = os.path.join(split_dir, cls_dir)
        if os.path.isdir(cls_path):
            imgs = glob.glob(os.path.join(cls_path, '*.jpg'))
            class_counts[cls_dir] = len(imgs)

    if not class_counts:
        return

    max_count = max(class_counts.values())
    print(f"\n  Target per kelas: {max_count}")

    aug = A.Compose([
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(0.3, 0.3, p=0.7),
        A.HueSaturationValue(15, 30, 25, p=0.7),
        A.GaussNoise(var_limit=(10, 40), p=0.3),
        A.GaussianBlur((3, 5), p=0.2),
        A.RandomGamma((80, 120), p=0.3),
        A.Affine(rotate=(-20, 20), scale=(0.8, 1.2), p=0.5),
    ])

    for cls_name, count in class_counts.items():
        need = max_count - count
        if need <= 0:
            print(f"  {cls_name:15s}: {count:4d} — OK")
            continue

        print(f"  {cls_name:15s}: {count:4d} → +{need} augmentasi", end="")
        cls_path = os.path.join(split_dir, cls_name)
        source_imgs = glob.glob(os.path.join(cls_path, '*.jpg'))

        generated = 0
        while generated < need:
            src = random.choice(source_imgs)
            img = cv2.imread(src)
            if img is None:
                continue
            result = aug(image=img)
            new_path = os.path.join(cls_path, f"aug_{generated}.jpg")
            cv2.imwrite(new_path, result['image'])
            generated += 1

        print(f" ✅")

print("⚖️  Balancing training crops...")
balance_crops(CROPS_DIR, 'train')
print("\n✅ Balancing selesai")

In [ ]:
# Visualisasi sampel crop per kelas
import matplotlib.pyplot as plt

train_dir = os.path.join(CROPS_DIR, 'train')
classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
n_classes = len(classes)
samples_per_class = 4

fig, axes = plt.subplots(n_classes, samples_per_class, figsize=(12, 3 * n_classes))
fig.suptitle('Sampel Crop per Kelas', fontsize=14, fontweight='bold')

for row, cls_name in enumerate(classes):
    cls_dir = os.path.join(train_dir, cls_name)
    imgs = glob.glob(os.path.join(cls_dir, '*.jpg'))
    samples = random.sample(imgs, min(samples_per_class, len(imgs)))

    for col in range(samples_per_class):
        ax = axes[row][col] if n_classes > 1 else axes[col]
        if col < len(samples):
            img = cv2.imread(samples[col])
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
        ax.axis('off')
        if col == 0:
            count = len(imgs)
            ax.set_title(f"{cls_name} ({count})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

## Langkah 3: Siapkan DataLoader

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Datasets
train_dataset = datasets.ImageFolder(os.path.join(CROPS_DIR, 'train'), transform=train_transform)
val_dir = os.path.join(CROPS_DIR, 'val')
has_val = os.path.exists(val_dir) and len(os.listdir(val_dir)) > 0

if has_val:
    val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)
else:
    # Split 80/20 from train
    from torch.utils.data import random_split
    total = len(train_dataset)
    val_size = int(total * 0.2)
    train_size = total - val_size
    train_dataset, val_dataset = random_split(train_dataset, [train_size, val_size])
    print(f"  Auto-split: {train_size} train, {val_size} val")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# Get class names from ImageFolder
if hasattr(train_dataset, 'classes'):
    FINAL_CLASSES = train_dataset.classes
elif hasattr(train_dataset, 'dataset'):
    FINAL_CLASSES = train_dataset.dataset.classes
else:
    FINAL_CLASSES = CLASS_NAMES

num_classes = len(FINAL_CLASSES)

print(f"\n📊 Dataset:")
print(f"  Train: {len(train_dataset)} gambar")
print(f"  Val  : {len(val_dataset)} gambar")
print(f"  Kelas: {num_classes} → {FINAL_CLASSES}")

## Langkah 4: Build & Train Model

In [ ]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import time
import copy

def build_model(arch, num_classes):
    """Build classifier model with pretrained weights."""
    if arch == 'efficientnet_v2_s':
        model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif arch == 'resnet50':
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif arch == 'mobilenet_v3_large':
        model = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    else:
        raise ValueError(f"Unknown arch: {arch}")
    return model

# Build
model = build_model(MODEL_ARCH, num_classes)
model = model.cuda()

# Count params
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n🏗️ Model: {MODEL_ARCH}")
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable:,}")
print(f"   Classes         : {num_classes}")

In [ ]:
# Training loop
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_acc = 0.0
best_model_wts = None
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"\n🏋️ Training {MODEL_ARCH} untuk {EPOCHS} epoch...\n")
start_time = time.time()

for epoch in range(EPOCHS):
    # === TRAIN ===
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.cuda(), labels.cuda()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # === VALIDATION ===
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.cuda(), labels.cuda()
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss_sum / val_total if val_total > 0 else 0
    val_acc = val_correct / val_total if val_total > 0 else 0

    scheduler.step()

    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    # Save best
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_wts = copy.deepcopy(model.state_dict())
        marker = ' ⭐ BEST'
    else:
        marker = ''

    if (epoch + 1) % 5 == 0 or epoch == 0 or marker:
        lr = optimizer.param_groups[0]['lr']
        print(f"  Epoch {epoch+1:3d}/{EPOCHS} | "
              f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} | "
              f"LR: {lr:.6f}{marker}")

elapsed = time.time() - start_time
print(f"\n✅ Training selesai dalam {elapsed/60:.1f} menit")
print(f"   Best Val Accuracy: {best_val_acc:.4f} ({best_val_acc*100:.1f}%)")

## Langkah 5: Evaluasi & Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Load best weights
model.load_state_dict(best_model_wts)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.cuda()
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Classification report
print("\n📊 Classification Report:\n")
print(classification_report(all_labels, all_preds, target_names=FINAL_CLASSES, digits=4))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', xticklabels=FINAL_CLASSES, yticklabels=FINAL_CLASSES, ax=ax1, cmap='Blues')
ax1.set_title('Confusion Matrix')
ax1.set_ylabel('Actual')
ax1.set_xlabel('Predicted')

sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=FINAL_CLASSES, yticklabels=FINAL_CLASSES, ax=ax2, cmap='Blues')
ax2.set_title('Confusion Matrix (Normalized)')
ax2.set_ylabel('Actual')
ax2.set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# Training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

ax1.plot(epochs_range, history['train_loss'], label='Train Loss')
ax1.plot(epochs_range, history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], label='Train Acc')
ax2.plot(epochs_range, history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training & Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n🎯 Best Validation Accuracy: {best_val_acc*100:.1f}%")

## Langkah 6: Export & Download `classifier.pt`

In [ ]:
from datetime import datetime

# Save checkpoint with metadata
checkpoint = {
    'model_state_dict': best_model_wts,
    'arch': MODEL_ARCH,
    'class_names': list(FINAL_CLASSES),
    'num_classes': num_classes,
    'image_size': IMAGE_SIZE,
    'best_val_acc': best_val_acc,
    'epochs_trained': EPOCHS,
    'timestamp': datetime.now().isoformat(),
}

# Save locally
local_path = f"{WORK_DIR}/classifier.pt"
torch.save(checkpoint, local_path)
size_mb = os.path.getsize(local_path) / (1024 * 1024)
print(f"💾 Classifier disimpan: {local_path} ({size_mb:.1f} MB)")

# Save to Drive
os.makedirs(OUTPUT_DRIVE, exist_ok=True)
drive_path = os.path.join(OUTPUT_DRIVE, 'classifier.pt')
shutil.copy2(local_path, drive_path)
print(f"💾 Disalin ke Drive: {drive_path}")

# Backup
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
backup_path = os.path.join(OUTPUT_DRIVE, f'classifier_{ts}.pt')
shutil.copy2(local_path, backup_path)
print(f"💾 Backup: {backup_path}")

print(f"\n📊 Ringkasan model:")
print(f"   Arsitektur   : {MODEL_ARCH}")
print(f"   Kelas        : {num_classes} → {list(FINAL_CLASSES)}")
print(f"   Ukuran       : {size_mb:.1f} MB")
print(f"   Val Accuracy : {best_val_acc*100:.1f}%")

In [ ]:
# Download classifier.pt ke laptop
from google.colab import files
print("📥 Downloading classifier.pt...")
files.download(local_path)

## 🔄 Cara Menggunakan classifier.pt

1. **Download** `classifier.pt` dari cell di atas
2. **Salin** ke folder GUI Duren:
   ```
   d:\GUI Duren\GUI Duren\classifier.pt
   ```
3. **Jalankan** GUI Duren seperti biasa:
   ```bash
   python main.py
   ```
4. **Otomatis aktif!** Di console akan muncul:
   ```
   [YOLOEngine] Two-stage mode AKTIF — classifier loaded
   ```

### Pipeline yang terjadi:
```
Frame capture
  → YOLO mendeteksi bounding box durian
    → Crop setiap durian dari frame
      → EfficientNet mengklasifikasi varietas
        → Hasil akhir ditampilkan di overlay
```

### Jika ingin kembali ke YOLO-only:
Cukup **hapus atau rename** `classifier.pt` — aplikasi otomatis fallback ke mode YOLO-only.

---

## (Opsional) Tes Inferensi Classifier

In [ ]:
# Test classifier pada gambar random dari validation set
model.load_state_dict(best_model_wts)
model.eval()

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
fig.suptitle('Tes Prediksi Classifier', fontsize=14, fontweight='bold')

# Get all val images
val_dir_path = os.path.join(CROPS_DIR, 'val')
test_images = []
test_labels = []

if os.path.exists(val_dir_path):
    for cls_name in sorted(os.listdir(val_dir_path)):
        cls_path = os.path.join(val_dir_path, cls_name)
        if os.path.isdir(cls_path):
            for img_f in glob.glob(os.path.join(cls_path, '*.jpg'))[:3]:
                test_images.append(img_f)
                test_labels.append(cls_name)

samples = list(zip(test_images, test_labels))
random.shuffle(samples)
samples = samples[:10]

for idx, (img_path, true_label) in enumerate(samples):
    row, col = idx // 5, idx % 5
    ax = axes[row][col]

    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Predict
    tensor = val_transform(transforms.ToPILImage()(img_rgb)).unsqueeze(0).cuda()
    with torch.no_grad():
        out = model(tensor)
        probs = torch.softmax(out, 1)
        conf, pred = torch.max(probs, 1)

    pred_label = FINAL_CLASSES[pred.item()]
    pred_conf = conf.item()
    correct = pred_label == true_label

    ax.imshow(img_rgb)
    color = 'green' if correct else 'red'
    ax.set_title(f"Pred: {pred_label} ({pred_conf*100:.0f}%)\nTrue: {true_label}",
                 color=color, fontsize=10)
    ax.axis('off')

plt.tight_layout()
plt.show()